# 2D weld pool: heat, flow, solidification

Version 1 for aluminium arc welding of a **monolithic plate** (joint gap not resolved).

**Process (physical):** GMAW / arc along the joint with filler wire.

**This model:**
- 2D longitudinal section \(x\) (along weld), \(z\) (thickness)
- Arc replaced by a **moving Gaussian surface heat flux**
- Liquid / solid via linear liquid fraction + Darcy sink in the mushy zone
- Driving forces: **Marangoni** + buoyancy (Boussinesq)
- **No** electromagnetism, arc pressure, free-surface deformation, or filler addition

**References** (`refs/weld_pool`):
- He, Fuerschbach, DebRoy, *J. Phys. D* 36 (2003) — transient heat/flow, Gaussian flux, Carman–Kozeny
- Zhao & DebRoy, *Metall. Mater. Trans. B* 32B (2001) — Al 5182 properties, negative \(d\sigma/dT\)
- Pitscheneder et al., *Welding Journal* (1996) — convection / Peclet controls pool shape
- Kidess et al., *Int. J. Thermal Sci.* 104 (2016) — laminar viscosity under-mixes (no effective-viscosity tuning here)


## Схема постановки (версия 1)

Продольное сечение монолитной пластины. Стык и зазор не моделируются.

```
        q(x,t)  Gaussian, moves with arc
              |||||  →  v_weld (+x)
    z=Lz  ─────████████████████──────────── top surface (Marangoni)
              │  liquid pool f_L≈1      │
              │  mushy  0<f_L<1        │
    z↑        │  solid  f_L=0          │
              │                        │
    z=0   ────┴────────────────────────┴──── bottom (convection)
         x=0 ──────────→ x ──────────────── Lx

  Sides: insulated;  no Lorentz / no filler / flat free surface
```

$$
x_c(t) = x_0 + v_{\mathrm{weld}}\, t,
\qquad
0 \le x \le L_x,\quad 0 \le z \le L_z
$$

$$
q = q_{\max}\exp\Big(-3\frac{(x-x_c)^2}{r_b^2}\Big)
- h(T-T_a) - \varepsilon\sigma(T^4-T_a^4)
$$

$$
\mu\partial_z u = f_L\,\frac{d\sigma}{dT}\,\partial_x T,
\qquad w=0
\quad\text{(top)}
$$

| Символ | Смысл | Типично |
|---|---|---|
| $x$ | вдоль шва | 0…30–40 mm |
| $z$ | толщина | 0…5–6 mm |
| $v_{\mathrm{weld}}$ | скорость дуги | $+x$ |
| $x_c(t)$ | центр пятна нагрева | $x_0 + v_{\mathrm{weld}} t$ |
| $q_{\max},\, r_b$ | пик и радиус гаусса | эффективные 2D |


## Equations

Continuity and momentum (He et al.):

$$
\nabla\cdot\mathbf{V} = 0
$$

$$
\rho \partial_t \mathbf{V}
= -\rho \nabla\cdot(\mathbf{V}\mathbf{V})
+ \nabla\cdot(\mu\nabla\mathbf{V})
- \nabla p
- C\frac{(1-f_L)^2}{f_L^3+B}\mathbf{V}
+ \rho g\beta(T-T_\mathrm{ref})\,\mathbf{e}_z
$$

Energy with apparent heat capacity in the mushy zone \(T_s < T < T_l\):

$$
\rho c_{p,\mathrm{eff}}\partial_t T
+ \rho c_p \mathbf{V}\cdot\nabla T
= \nabla\cdot(k\nabla T),
\qquad
c_{p,\mathrm{eff}} = c_p + \frac{L}{T_l-T_s}\ \text{(in mush)}
$$

$$
f_L = \mathrm{clip}\big((T-T_s)/(T_l-T_s),\,0,\,1\big)
$$

Top surface:

$$
q = q_{\max}\exp\Big(-3\frac{(x-x_0-vt)^2}{r_b^2}\Big)
- h(T-T_a) - \varepsilon\sigma(T^4-T_a^4)
$$

$$
\mu\partial_z u = f_L\,\frac{d\sigma}{dT}\,\partial_x T,
\qquad w=0
$$

Sides insulated; bottom cooled by convection. Power is **effective 2D** (per unit width), not a full 3D arc passport.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name != 'heat+flow_2d':
    cand = ROOT / 'heat+flow_2d'
    if cand.is_dir():
        sys.path.insert(0, str(cand))
        MODEL_DIR = cand
    else:
        sys.path.insert(0, str(ROOT))
        MODEL_DIR = ROOT
else:
    sys.path.insert(0, str(ROOT))
    MODEL_DIR = ROOT

from weldpool2d import Material, Process, WeldPool2D, quick_process

print('model dir:', MODEL_DIR)


## Material (Al 5182) and process parameters

Defaults from Zhao & DebRoy (2001), Table III.  
QUICK = True runs a coarse short case (~minutes of CPU → seconds here). Set False for the finer default grid.


In [ ]:
QUICK = True  # set False for finer Process() defaults

mat = Material()  # Al 5182

if QUICK:
    proc = quick_process(
        enable_flow=True,
        t_end=1.2,
        q_max=8.0e7,   # W/m^2 peak (effective 2D)
        r_b=0.0025,
        v_weld=0.008,
        x0=0.006,
    )
else:
    proc = Process(
        enable_flow=True,
        t_end=3.0,
        q_max=6.0e7,
        r_b=0.003,
        v_weld=0.005,
        x0=0.008,
        nx=120,
        nz=30,
    )

print('grid', proc.nx, 'x', proc.nz, '  Lx,Lz [mm] =', 1e3*proc.Lx, 1e3*proc.Lz)
print('q_max =', proc.q_max, 'W/m^2   rb =', proc.r_b, 'm   v =', proc.v_weld, 'm/s')
print('Ts,Tl =', mat.T_solidus, mat.T_liquidus, 'K   dsigma/dT =', mat.dsigma_dT)


## Run with flow

In [ ]:
model = WeldPool2D(material=mat, process=proc)
diag = model.run(record_every=0.05, progress=True, record_fields=True)

print('heat balance |Qin - Qloss - dH| / Qin =', round(model.heat_balance_ratio(), 4))
print('final Tmax =', round(model.T.max(), 1), 'K')
depth, length = model.pool_metrics()
print('pool depth, length [mm] =', round(1e3*depth, 2), round(1e3*length, 2))
print('max |u| =', round(max(diag.max_speed), 4), 'm/s   max Pe =', round(max(diag.pe_number), 2))
print('T_max frames recorded:', len(diag.T_max_frames))


## Fields: temperature, liquid fraction, velocity

In [ ]:
def plot_fields(m, title_prefix=''):
    Xmm = 1e3 * m.X
    Zmm = 1e3 * m.Z
    u_c, w_c = m._cell_velocity()
    fig, axs = plt.subplots(1, 3, figsize=(14, 3.6), constrained_layout=True)

    im0 = axs[0].pcolormesh(Xmm, Zmm, m.T, shading='auto', cmap='inferno')
    axs[0].contour(Xmm, Zmm, m.T, levels=[m.mat.T_solidus, m.mat.T_liquidus], colors=['cyan', 'lime'], linewidths=1.2)
    axs[0].axvline(1e3 * m.arc_centre(), color='w', ls='--', lw=0.8)
    fig.colorbar(im0, ax=axs[0], label='T [K]')
    axs[0].set_title(f'{title_prefix}Temperature')

    im1 = axs[1].pcolormesh(Xmm, Zmm, m.fL, shading='auto', cmap='Blues', vmin=0, vmax=1)
    fig.colorbar(im1, ax=axs[1], label='f_L')
    axs[1].set_title('Liquid fraction')

    speed = np.sqrt(u_c**2 + w_c**2)
    im2 = axs[2].pcolormesh(Xmm, Zmm, speed, shading='auto', cmap='viridis')
    # quiver subsample
    sx, sz = max(1, m.proc.nx // 25), max(1, m.proc.nz // 10)
    axs[2].quiver(
        Xmm[::sx, ::sz], Zmm[::sx, ::sz],
        u_c[::sx, ::sz], w_c[::sx, ::sz],
        color='w', scale=None, width=0.003,
    )
    fig.colorbar(im2, ax=axs[2], label='|V| [m/s]')
    axs[2].set_title('Velocity')

    for ax in axs:
        ax.set_xlabel('x [mm]')
        ax.set_ylabel('z [mm]')
        ax.set_aspect('equal')
    return fig

plot_fields(model, title_prefix=f't={model.t:.2f}s  ')
plt.show()


## Velocity

Separate field of `|V|` (arrows only where `f_L > 0.3`) and the recorded maximum speed.


In [ ]:
def plot_velocity(m, diag, title_prefix=''):
    Xmm = 1e3 * m.X
    Zmm = 1e3 * m.Z
    u_c, w_c = m._cell_velocity()
    speed = np.sqrt(u_c**2 + w_c**2)

    fig, axs = plt.subplots(
        2, 1, figsize=(12, 5.6),
        constrained_layout=True,
        gridspec_kw={'height_ratios': [1.6, 1]},
    )

    ax = axs[0]
    ax.set_aspect('equal')
    im = ax.pcolormesh(Xmm, Zmm, speed, shading='auto', cmap='viridis')
    ax.contour(Xmm, Zmm, m.T, levels=[m.mat.T_liquidus], colors='cyan', linewidths=1.0)
    ax.axvline(1e3 * m.arc_centre(), color='w', ls='--', lw=0.8)

    mask = m.fL > 0.3
    sx, sz = max(1, m.proc.nx // 25), max(1, m.proc.nz // 8)
    Xs, Zs = Xmm[::sx, ::sz], Zmm[::sx, ::sz]
    us, ws = u_c[::sx, ::sz], w_c[::sx, ::sz]
    ms = mask[::sx, ::sz]
    # Autoscale uses the wide x-axis, so arrows span the plate. Tie the
    # longest arrow to a fraction of the axes height (plate thickness).
    vmax = float(np.max(speed[mask])) if np.any(mask) else float(np.max(speed))
    vmax = max(vmax, 1e-9)
    q = ax.quiver(
        Xs[ms], Zs[ms], us[ms], ws[ms],
        color='k', angles='uv', scale_units='height', scale=vmax / 0.12,
        width=0.004, pivot='mid',
    )
    ax.quiverkey(q, 0.86, 1.08, 0.2, '0.2 m/s', labelpos='E', coordinates='axes')
    fig.colorbar(im, ax=ax, label='|V| [m/s]')
    ax.set_title(f'{title_prefix}Velocity   max |V| = {speed.max():.3f} m/s')
    ax.set_xlabel('x [mm]')
    ax.set_ylabel('z [mm]')
    ax.set_aspect('equal')

    t = np.array(diag.time)
    axs[1].plot(t, diag.max_speed, color='C2')
    axs[1].set_xlabel('t [s]')
    axs[1].set_ylabel('|V|_max [m/s]')
    axs[1].set_title('Maximum speed')
    axs[1].grid(True, alpha=0.3)
    return fig

plot_velocity(model, diag, title_prefix=f't={model.t:.2f}s  ')
plt.show()


## Pool size and thermal cycle


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
t = np.array(diag.time)

axs[0].plot(t, 1e3 * np.array(diag.pool_depth), label='depth')
axs[0].plot(t, 1e3 * np.array(diag.pool_length), label='length')
axs[0].set_xlabel('t [s]'); axs[0].set_ylabel('mm'); axs[0].legend(); axs[0].set_title('Pool size')
axs[0].grid(True, alpha=0.3)

axs[1].plot(t, diag.T_probe, color='C1')
axs[1].axhline(mat.T_solidus, color='c', ls='--', label='Ts')
axs[1].axhline(mat.T_liquidus, color='g', ls='--', label='Tl')
axs[1].set_xlabel('t [s]'); axs[1].set_ylabel('T [K]'); axs[1].legend()
axs[1].set_title(f'Thermocycle at probe {model.probe_ij}'); axs[1].grid(True, alpha=0.3)
plt.show()


## Fusion zone (T_max > Tl)

Final cumulative max temperature and contour of the fusion zone, then animation of how the zone grows with time.


In [ ]:
Xmm = 1e3 * model.X
Zmm = 1e3 * model.Z
fz = model.T_max >= mat.T_liquidus

fig, ax = plt.subplots(figsize=(8, 3.2), constrained_layout=True)
im = ax.pcolormesh(Xmm, Zmm, model.T_max, shading='auto', cmap='hot')
ax.contour(Xmm, Zmm, fz.astype(float), levels=[0.5], colors='cyan')
fig.colorbar(im, ax=ax, label='T_max [K]')
ax.set_title('Fusion zone (T_max > Tl)')
ax.set_xlabel('x [mm]')
ax.set_ylabel('z [mm]')
ax.set_aspect('equal')
plt.show()


In [ ]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

assert diag.T_max_frames, 'No T_max frames — re-run with record_fields=True'

t = np.array(diag.time)
frames = diag.T_max_frames
vmin = float(min(f.min() for f in frames))
vmax = float(max(f.max() for f in frames))
Xmm = 1e3 * model.X
Zmm = 1e3 * model.Z

fig, ax = plt.subplots(figsize=(8, 3.2), constrained_layout=True)
im = ax.pcolormesh(Xmm, Zmm, frames[0], shading='auto', cmap='hot', vmin=vmin, vmax=vmax)
cs = ax.contour(
    Xmm, Zmm, (frames[0] >= mat.T_liquidus).astype(float),
    levels=[0.5], colors='cyan',
)
arc_line = ax.axvline(1e3 * model.arc_centre(float(t[0])), color='w', ls='--', lw=0.8)
fig.colorbar(im, ax=ax, label='T_max [K]')
ax.set_xlabel('x [mm]')
ax.set_ylabel('z [mm]')
ax.set_aspect('equal')
title = ax.set_title(f'Fusion zone  t = {t[0]:.2f} s')


def _remove_contour(cset):
    if cset is None:
        return
    if hasattr(cset, 'remove'):
        cset.remove()
        return
    for coll in getattr(cset, 'collections', []):
        coll.remove()


def update(i):
    global cs
    im.set_array(frames[i].ravel())
    _remove_contour(cs)
    cs = ax.contour(
        Xmm, Zmm, (frames[i] >= mat.T_liquidus).astype(float),
        levels=[0.5], colors='cyan',
    )
    xc = 1e3 * model.arc_centre(float(t[i]))
    arc_line.set_xdata([xc, xc])
    title.set_text(f'Fusion zone  t = {t[i]:.2f} s')
    return [im, arc_line, title]


anim = FuncAnimation(fig, update, frames=len(frames), interval=120, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())


## Checks

1. Heat balance (He-style): \(|Q_\mathrm{in}-Q_\mathrm{loss}-\Delta H|/Q_\mathrm{in}\)
2. Velocity ≈ 0 in solid (\(f_L=0\))
3. Marangoni: for \(d\sigma/dT<0\), surface flow **away** from the hot spot
4. Optional: conduction-only vs with flow (pool depth)


In [ ]:
u_c, w_c = model._cell_velocity()
speed = np.sqrt(u_c**2 + w_c**2)
solid = model.fL < 1e-6
liquid = model.fL > 0.5
print('=== Heat balance ===')
print('ratio =', round(model.heat_balance_ratio(), 4))

print('=== Solid velocity ===')
print('max |V| in solid =', float(speed[solid].max()) if np.any(solid) else 0.0)
print('max |V| in liquid =', float(speed[liquid].max()) if np.any(liquid) else 0.0)

print('=== Marangoni direction (top surface) ===')
xc = model.arc_centre()
u_top = model.u[:, -1]
# mean u in a window left/right of arc where surface is molten
fL_face = np.zeros_like(u_top)
fL_face[1:-1] = 0.5 * (model.fL[:-1, -1] + model.fL[1:, -1])
left = (model.xu < xc) & (fL_face > 0.3)
right = (model.xu > xc) & (fL_face > 0.3)
u_left = float(u_top[left].mean()) if np.any(left) else float('nan')
u_right = float(u_top[right].mean()) if np.any(right) else float('nan')
print('mean u left of arc =', round(u_left, 4), ' (expect < 0)')
print('mean u right of arc =', round(u_right, 4), ' (expect > 0)')
print('dsigma/dT =', mat.dsigma_dT, '< 0 => outflow from centre: OK' if (u_left < 0 and u_right > 0) else '')


In [ ]:
# Conduction-only comparison (same process, short)
proc_cond = quick_process(
    enable_flow=False,
    t_end=min(0.8, proc.t_end),
    q_max=proc.q_max,
    r_b=proc.r_b,
    v_weld=proc.v_weld,
    x0=proc.x0,
    nx=proc.nx,
    nz=proc.nz,
    Lx=proc.Lx,
    Lz=proc.Lz,
)
m_cond = WeldPool2D(material=mat, process=proc_cond)
m_cond.run(record_every=0.1, progress=False)
d_flow, _ = model.pool_metrics()
# re-run flow to same time for fair compare
proc_flow = quick_process(
    enable_flow=True,
    t_end=proc_cond.t_end,
    q_max=proc.q_max,
    r_b=proc.r_b,
    v_weld=proc.v_weld,
    x0=proc.x0,
    nx=proc.nx,
    nz=proc.nz,
    Lx=proc.Lx,
    Lz=proc.Lz,
)
m_flow = WeldPool2D(material=mat, process=proc_flow)
m_flow.run(record_every=0.1, progress=False)
d_c, L_c = m_cond.pool_metrics()
d_f, L_f = m_flow.pool_metrics()
print(f'At t={proc_cond.t_end}s:')
print(f'  conduction-only: depth={1e3*d_c:.2f} mm  length={1e3*L_c:.2f} mm  Tmax={m_cond.T.max():.0f} K')
print(f'  with flow:       depth={1e3*d_f:.2f} mm  length={1e3*L_f:.2f} mm  Tmax={m_flow.T.max():.0f} K')
print(f'  Pe (flow) = {m_flow.peclet():.1f}')


## Notes / next versions

- Filler wire / droplet heat and mass not included (monolithic plate).
- No Lorentz force — add MHD in a later version.
- Flat free surface; no arc pressure.
- Molecular viscosity only (Kidess: underestimates mixing).
- 2D omits lateral heat loss: calibrate \(q_{\max}, r_b\) as effective parameters.
